# So sánh Pretrained vs Fine-tuned
So sánh model gốc (COCO) với model đã fine-tune trên dataset bóng đá.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if not os.path.exists('/content/football_tracking'):
    !git clone https://github.com/henruysun2511/football_tracking.git /content/football_tracking
%cd /content/football_tracking
!pip install -q ultralytics pyyaml matplotlib pandas

In [ ]:
from roboflow import Roboflow
from pathlib import Path

api_key = input("ROBOFLOW_API_KEY: ")
rf = Roboflow(api_key=api_key)

if not os.path.exists('datasets/football-players-detection-2'):
    project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
    dataset = project.version(2).download("yolov8")
    !mv {dataset.location} datasets/football-players-detection-2

if os.path.exists('/content/drive/MyDrive/football_models/player_detector.pt'):
    !cp /content/drive/MyDrive/football_models/player_detector.pt models/
    print("Copied from Drive")
elif not os.path.exists('models/player_detector.pt'):
    print("WARNING: No fine-tuned model found!")

In [ ]:
import os, yaml, warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

warnings.filterwarnings('ignore')
ROOT = Path(os.getcwd())
OUTPUT_PNG = "/content/drive/MyDrive/football_models/comparison.png"
os.makedirs("/content/drive/MyDrive/football_models", exist_ok=True)

print("="*60)
print("  SO SÁNH MODEL: PRETRAINED (COCO) vs FINE-TUNED (Football)")
print("="*60)

print("\n[1] Loading models...")
model_ft = YOLO("models/player_detector.pt")

In [ ]:
# ─── 2. Thông số kiến trúc ───
print("\n" + "-"*60)
print("  BẢNG 1 — THÔNG SỐ KIẾN TRÚC")
print("-"*60)

params = sum(p.numel() for p in model_ft.model.parameters())

print(f"  {'Model':<20} {'Classes':<10} {'Params':<15} {'GFLOPs':<10}")
print(f"  {'-'*20} {'-'*10} {'-'*15} {'-'*10}")
print(f"  {'YOLOv8x (COCO)':<20} {80:<10} {68.2:<15.2f}M {257.8:<10.1f}")
print(f"  {'YOLOv8x (Fine-tuned)':<20} {4:<10} {params/1e6:<15.2f}M {257.8:<10.1f}")

In [ ]:
# ─── 3. Fix data.yaml ───
base = ROOT / "datasets" / "football-players-detection-2"
with open(base / "data.yaml") as f:
    data_cfg = yaml.safe_load(f)
data_cfg["train"] = str(base / "train" / "images")
data_cfg["val"]   = str(base / "valid" / "images")
fixed_yaml = str(base / "_data_fixed.yaml")
with open(fixed_yaml, "w") as f:
    yaml.dump(data_cfg, f)

In [ ]:
# ─── 4. Val fine-tuned model ───
print("\n" + "-"*60)
print("  EVAL FINE-TUNED → Football Dataset")
print("-"*60)
results_ft = model_ft.val(data=fixed_yaml, imgsz=1280, batch=8, device=0, plots=False, verbose=False)

ft_map50    = float(results_ft.box.map50)
ft_map50_95 = float(results_ft.box.map)
ft_prec     = float(results_ft.box.p[0]) if hasattr(results_ft.box, 'p') and len(results_ft.box.p) > 0 else 0
ft_rec      = float(results_ft.box.r[0]) if hasattr(results_ft.box, 'r') and len(results_ft.box.r) > 0 else 0
ft_ap50     = results_ft.box.ap50.tolist() if hasattr(results_ft.box, 'ap50') else [0]*4

print(f"  mAP@0.5    : {ft_map50:.4f}")
print(f"  mAP@0.5:0.95: {ft_map50_95:.4f}")
print(f"  Precision  : {ft_prec:.4f}")
print(f"  Recall     : {ft_rec:.4f}")

CLASSES = ["ball", "goalkeeper", "player", "referee"]
for i, name in enumerate(CLASSES):
    print(f"  {name:<15}: mAP@0.5 = {ft_ap50[i]:.4f}")

In [ ]:
# ─── 5. Baseline COCO (số liệu chuẩn từ Ultralytics) ───
print("\n" + "="*65)
print("  SO SÁNH VỚI COCO BASELINE")
print("="*65)
# Nguồn: https://docs.ultralytics.com/models/yolov8/#performance
# YOLOv8x trên COCO val2017 (imgsz=640, 80 classes)
print("")
print("  YOLOv8x trên COCO val2017 (nguồn: Ultralytics docs):")
print(f"    mAP@0.5     : 0.685")
print(f"    mAP@0.5:0.95: 0.539")
print("")
print("  YOLOv8x fine-tuned trên Football dataset:")
print(f"    mAP@0.5     : {ft_map50:.4f}")
print(f"    mAP@0.5:0.95: {ft_map50_95:.4f}")
print("")
print("  Lưu ý: Hai số liệu này không so sánh trực tiếp vì khác dataset")
print("  (80 classes COCO vs 4 classes football) và khác imgsz (640 vs 1280).")

In [ ]:
# ─── 6. Biểu đồ kết quả fine-tuned ───
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("YOLOv8x Fine-tuned — Football Dataset Performance",
             fontsize=13, fontweight="bold")

# (a) Overall metrics
ax = axes[0]
metrics = ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall"]
vals = [ft_map50, ft_map50_95, ft_prec, ft_rec]
colors = ["#2ecc71", "#27ae60", "#3498db", "#9b59b6"]
bars = ax.bar(metrics, vals, color=colors, edgecolor="white", width=0.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{val:.3f}", ha="center", va="bottom", fontweight="bold")
ax.set_ylim(0, 1.15)
ax.grid(alpha=0.3, axis="y")
ax.set_title("Overall Metrics")

# (b) Per-class mAP@0.5
ax = axes[1]
class_colors = ["#e74c3c", "#f39c12", "#2ecc71", "#3498db"]
bars = ax.bar(CLASSES, ft_ap50, color=class_colors, edgecolor="white", width=0.5)
for bar, val in zip(bars, ft_ap50):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{val:.3f}", ha="center", va="bottom", fontweight="bold")
ax.set_ylim(0, 1.15)
ax.grid(alpha=0.3, axis="y")
ax.set_title("Per-class mAP@0.5")

plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUTPUT_PNG}")

os.remove(fixed_yaml)